# Agenter ACP Backend with OpenAI Codex

This notebook demonstrates Agenter acting as a real ACP client. It connects to a real ACP-compatible OpenAI Codex adapter process, then runs a coding task through Agenter's normal validation and result pipeline.

Flow:

```text
Agenter AutonomousCodingAgent
  -> ACPBackend
  -> codex-acp adapter
  -> OpenAI Codex
```

## Requirements

Install Agenter with ACP support:

```bash
pip install -e ".[acp]"
```

Install a real ACP adapter for OpenAI Codex. Either install the binary globally:

```bash
npm install -g @zed-industries/codex-acp
```

Set one of the credentials supported by the adapter, usually:

```bash
export OPENAI_API_KEY="..."
```

You can optionally set `CODEX_MODEL` and `CODEX_REASONING_EFFORT`; the notebook passes both through as Codex config overrides.

In [1]:
# Uncomment if this notebook kernel does not have the Python dependency installed.
# %pip install -e ".[acp]"

In [1]:
import os
import shutil
import tempfile
from pathlib import Path

from agenter import AutonomousCodingAgent, CodingRequest, Verbosity, configure_logging

configure_logging(quiet=True)

CODEX_MODEL = os.getenv("CODEX_MODEL", "gpt-5.4")
CODEX_REASONING_EFFORT = os.getenv("CODEX_REASONING_EFFORT", "high")
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))
print("Codex model:", CODEX_MODEL)
print("Codex reasoning effort:", CODEX_REASONING_EFFORT)

OPENAI_API_KEY set: True
Codex model: gpt-5.4
Codex reasoning effort: high


## Resolve the Real ACP Agent Command

This mirrors the other external-runtime backends: the command must already be installed and available on `PATH`.

In [2]:
acp_command = shutil.which("codex-acp")

if acp_command is None:
    raise RuntimeError("Install codex-acp first: npm install -g @zed-industries/codex-acp")

print("ACP command:", acp_command)

ACP command: /opt/homebrew/bin/codex-acp


## Run Agenter Through ACP

This cell launches the real Codex ACP adapter and asks it to modify a temporary workspace. Agenter observes the changed files and validates the Python output.

In [3]:
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY before running this cell.")

workspace = Path(tempfile.mkdtemp(prefix="agenter-codex-acp-workspace-"))

agent = AutonomousCodingAgent(
    backend="acp",
    acp_command=acp_command,
    acp_args=[
        "-c",
        f'model="{CODEX_MODEL}"',
        "-c",
        f'model_reasoning_effort="{CODEX_REASONING_EFFORT}"',
        "-c",
        'approval_policy="never"',
        "-c",
        'sandbox_mode="workspace-write"',
    ],
    acp_env={
        "OPENAI_API_KEY": os.environ["OPENAI_API_KEY"],
    },
    acp_permission_policy="allow",
)

result = await agent.execute(
    CodingRequest(
        prompt=(
            "Create a file named math_tools.py with an add(a, b) function, "
            "a multiply(a, b) function, and a small __main__ demo. "
            "Write the file directly in the current workspace."
        ),
        cwd=str(workspace),
        max_iterations=2,
    ),
    verbosity=Verbosity.QUIET,
)

if "math_tools.py" not in result.files:
    raise AssertionError(f"Expected math_tools.py in result.files, got {list(result.files)}")

# The next cell prints the result in a stable, notebook-friendly format.

In [4]:
print("status:", result.status.value)
print("workspace:", workspace)
print("files:", list(result.files))

for path, content in result.files.items():
    print(f"\n--- {path} ---")
    print(content)

shutil.rmtree(workspace, ignore_errors=True)
print("\nworkspace cleaned:", not workspace.exists())

status: completed
workspace: /var/folders/0f/1wtyzkj508nbh_hny1k3rrbm0000gn/T/agenter-codex-acp-workspace-7jljvy5w
files: ['math_tools.py']

--- math_tools.py ---
"""Simple math helper functions with a small command-line demo."""


def add(a, b):
    """Return the sum of two values."""
    return a + b


def multiply(a, b):
    """Return the product of two values."""
    return a * b


if __name__ == "__main__":
    first = 3
    second = 4

    print(f"add({first}, {second}) = {add(first, second)}")
    print(f"multiply({first}, {second}) = {multiply(first, second)}")


workspace cleaned: True
